# 02 · Fase A — Teardown de DETR-R50
Parámetros y FLOPs por bloque, latencia/memoria, características intermedias, mapas de atención, campo receptivo, línea base en COCO y **costo en función de N (H3)**.
*FLOPs = 2 × MACs (convención de `torch.utils.flop_counter`).*

In [ ]:
# Configuración común (Colab/Kaggle o local). Ajusta REPO_URL una sola vez.
import os, sys
REPO_URL = 'https://github.com/<usuario>/detr-query-saturation.git'
if not os.path.exists('../src/common.py'):            # no estamos dentro de notebooks/ -> clonar
    if not os.path.exists('detr-query-saturation/src'):
        !git clone -q {REPO_URL}
    os.chdir('detr-query-saturation/notebooks')
    !pip -q install -r ../requirements.txt
sys.path.insert(0, '../src')
from common import *
import torch, json, pandas as pd, matplotlib.pyplot as plt
os.makedirs('../results', exist_ok=True); os.makedirs('../data', exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'; SEED = 42; set_seed(SEED)
print(DEVICE); print(env_report('../results/entorno.txt'))   # versiones exactas -> README

In [ ]:
# Rutas de la copia de Kaggle de VisDrone (ajusta tras revisar el árbol en 01)
ROOT = '/kaggle/input/<dataset-visdrone>'
TRAIN_IMG, TRAIN_LBL = f'{ROOT}/<train/images>', f'{ROOT}/<train/labels>'
VAL_IMG,   VAL_LBL   = f'{ROOT}/<val/images>',   f'{ROOT}/<val/labels>'

from transformers import DetrForObjectDetection, DetrImageProcessor
from PIL import Image
CKPT = 'facebook/detr-resnet-50'
processor = DetrImageProcessor.from_pretrained(CKPT)          # resolución original (lado corto 800)
model = DetrForObjectDetection.from_pretrained(CKPT).to(DEVICE).eval()
# Imagen de ejemplo: la más densa de validación
cnt = objects_per_image('../data/val_sub.json'); vd = json.load(open('../data/val_sub.json'))
dense = max(vd['images'], key=lambda im: cnt[im['id']]); print(dense['file_name'], cnt[dense['id']], 'objetos')
img = Image.open(os.path.join(VAL_IMG, dense['file_name'])).convert('RGB'); enc = processor(images=img, return_tensors='pt')
pv, pm = enc['pixel_values'].to(DEVICE), enc['pixel_mask'].to(DEVICE); print(pv.shape)

## A1 · Parámetros por bloque

In [ ]:
blocks = {'backbone (ResNet-50)': model.model.backbone, 'input_projection': model.model.input_projection,
          'query_embeddings': model.model.query_position_embeddings, 'encoder (6 capas)': model.model.encoder,
          'decoder (6 capas)': model.model.decoder, 'class head': model.class_labels_classifier, 'box head (MLP)': model.bbox_predictor}
P = pd.DataFrame([(k, sum(p.numel() for p in m.parameters())) for k, m in blocks.items()], columns=['bloque', 'parámetros'])
P['%'] = (100 * P['parámetros'] / P['parámetros'].sum()).round(1); print(P)
print('suma bloques', P['parámetros'].sum(), '| total modelo', count_params(model))   # deben coincidir
P.to_csv('../results/params_por_bloque.csv', index=False)

## A2 · FLOPs por bloque

In [ ]:
from torch.utils.flop_counter import FlopCounterMode
with torch.no_grad(), FlopCounterMode(model, depth=4, display=False) as fc: model(pixel_values=pv, pixel_mask=pm)
counts = {k: sum(v.values()) for k, v in fc.get_flop_counts().items()}
for k, v in counts.items(): print(f'{v/1e9:9.2f} GFLOPs  {k}')   # revisa los nombres y ajusta los filtros si hace falta
rows = []
for key in ['model.backbone', 'model.input_projection', 'model.encoder', 'model.decoder', 'class_labels_classifier', 'bbox_predictor']:
    rows.append((key, sum(v for k, v in counts.items() if k.endswith(key)) / 1e9))
total = fc.get_total_flops() / 1e9
F = pd.DataFrame(rows, columns=['bloque', 'GFLOPs']); F['%'] = (100 * F['GFLOPs'] / total).round(1)
print(F, '\ntotal', round(total, 2), 'GFLOPs | suma de bloques', round(F['GFLOPs'].sum(), 2), '(la diferencia: pos. embeddings y ops sueltas)')
F.to_csv('../results/flops_por_bloque.csv', index=False)

## A3 · Latencia y memoria (calentamiento + repeticiones)

In [ ]:
import time
assert DEVICE == 'cuda', 'medir latencia en GPU'
torch.cuda.reset_peak_memory_stats()
lat = latency_ms(model, pv, pm, runs=30, warmup=10)
# por bloque (aprox.): hooks con sincronización; se retiran al terminar
times, handles = {}, []
def mk(name):
    def pre(m, i): torch.cuda.synchronize(); times[name] = -time.perf_counter()
    def post(m, i, o): torch.cuda.synchronize(); times[name] += time.perf_counter()
    return pre, post
for name, m in {'backbone': model.model.backbone, 'encoder': model.model.encoder, 'decoder': model.model.decoder}.items():
    a, b = mk(name); handles += [m.register_forward_pre_hook(a), m.register_forward_hook(b)]
with torch.no_grad(): model(pixel_values=pv, pixel_mask=pm)
for h in handles: h.remove()
print(f'latencia total {lat:.1f} ms | memoria pico {torch.cuda.max_memory_allocated()/2**20:.0f} MiB (incluye pesos)')
print({k: round(v * 1000, 1) for k, v in times.items()}, 'ms (una pasada, aprox.)')
pd.DataFrame([{'latencia_ms': lat, 'mem_pico_MiB': torch.cuda.max_memory_allocated() / 2**20,
               **{f'{k}_ms': v * 1000 for k, v in times.items()}, 'entrada': tuple(pv.shape)}]).to_csv('../results/latencia_memoria.csv', index=False)

## A4 · Características intermedias del backbone

In [ ]:
with torch.no_grad(): feats = model.model.backbone.conv_encoder(pv, pm)
fig, ax = plt.subplots(1, len(feats) + 1, figsize=(4 * (len(feats) + 1), 4)); ax[0].imshow(img); ax[0].set_title('imagen')
for i, (f, _) in enumerate(feats): ax[i + 1].imshow(f[0].mean(0).cpu(), cmap='viridis'); ax[i + 1].set_title(f'etapa {i+1}: {tuple(f.shape[1:])}')
for a in ax: a.axis('off')
plt.savefig('../results/features_intermedias.png', dpi=150, bbox_inches='tight')

## A5 · Mapas de atención del decoder (última capa, queries con mayor score)

In [ ]:
with torch.no_grad(): out = model(pixel_values=pv, pixel_mask=pm, output_attentions=True)
(features, _) = model.model.backbone(pv, pm); h, w = features[-1][0].shape[-2:]
att = out.cross_attentions[-1][0].mean(0)             # (queries, h*w), promedio de cabezas
top = out.logits[0].softmax(-1)[:, :-1].max(-1).values.topk(5).indices.tolist()
fig, ax = plt.subplots(1, 5, figsize=(20, 4))
for a, q in zip(ax, top): a.imshow(att[q].reshape(h, w).cpu()); a.set_title(f'query {q}'); a.axis('off')
plt.savefig('../results/atencion_decoder.png', dpi=150, bbox_inches='tight')

## A6 · Campo receptivo: efectivo (por gradiente) vs teórico
Teórico en la salida de `layer4` (stride 32): **427 px**. El backbone de DETR en Hugging Face es el `resnet50` de timm (ResNet v1.5: stride en la conv 3×3).
El valor de 483 px que se cita a veces corresponde a ResNet v1 (stride en la primera 1×1). Cálculo: r ← r + (k−1)·j, j ← j·s, capa por capa.

In [ ]:
def rf_resnet50(stride_on_3x3=True):
    r, j = 1, 1
    for k, s in [(7, 2), (3, 2)]: r += (k - 1) * j; j *= s           # conv1, maxpool
    for blocks, stride in [(3, 1), (4, 2), (6, 2), (3, 2)]:
        for b in range(blocks):
            s = stride if b == 0 else 1
            for k, st in ([(1, 1), (3, s), (1, 1)] if stride_on_3x3 else [(1, s), (3, 1), (1, 1)]):
                r += (k - 1) * j; j *= st
    return r, j
print('teórico v1.5 (timm):', rf_resnet50(True), '| v1:', rf_resnet50(False))   # (427, 32) y (483, 32)

x = pv.clone().requires_grad_(True)
fm = model.model.backbone(x, pm)[0][-1][0]; cy, cx = fm.shape[-2] // 2, fm.shape[-1] // 2
fm[0, :, cy, cx].sum().backward()
erf = x.grad.abs().sum(1)[0].cpu(); plt.imshow(erf, cmap='magma'); plt.title('campo receptivo efectivo (centro del mapa stride-32)')
plt.savefig('../results/campo_receptivo.png', dpi=150, bbox_inches='tight')
ys, xs = torch.where(erf > erf.max() * 0.05); print('extensión efectiva (>5% del máx) ≈', (ys.max() - ys.min()).item(), 'x', (xs.max() - xs.min()).item(), 'px  vs teórico 427')

## A7 · Reproducción de la línea base en COCO val2017 (subconjunto)
Descarga val2017 + anotaciones (Colab) y evalúa 500 imágenes. Referencia: AP = 42.0 en el val completo (Carion et al., 2020). Documenta la diferencia (subconjunto de 500 → varía unos puntos).

In [ ]:
# !wget -q http://images.cocodataset.org/zips/val2017.zip && unzip -q val2017.zip -d ../data
# !wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip && unzip -q annotations_trainval2017.zip -d ../data
print(make_subset('../data/annotations/instances_val2017.json', 500, '../data/coco_val_sub.json', SEED))
dets = predict(model, processor, '../data/val2017', '../data/coco_val_sub.json', device=DEVICE, label_offset=0)   # etiquetas = ids COCO
base = coco_eval('../data/coco_val_sub.json', dets); print({k: round(v, 3) for k, v in base.items()})
pd.DataFrame([base]).to_csv('../results/linea_base_coco.csv', index=False)

## A8 · Costo en función del número de queries (H3)
Mismo input que la ablación (lado corto 512). Solo cambia N; los pesos no importan para el costo.

In [ ]:
proc512 = DetrImageProcessor.from_pretrained(CKPT, size={'shortest_edge': 512, 'longest_edge': 853})
e = proc512(images=img, return_tensors='pt'); pv5, pm5 = e['pixel_values'].to(DEVICE), e['pixel_mask'].to(DEVICE)
rows = []
for N in (100, 300, 500):
    m = build_model(num_queries=N, num_labels=10).to(DEVICE).eval()
    torch.cuda.reset_peak_memory_stats()
    rows.append({'N': N, 'params_M': count_params(m) / 1e6, 'GFLOPs': flops_total(m, pv5, pm5) / 1e9,
                 'latencia_ms': latency_ms(m, pv5, pm5), 'mem_pico_MiB': torch.cuda.max_memory_allocated() / 2**20})
    del m; torch.cuda.empty_cache()
H3 = pd.DataFrame(rows); H3['ΔGFLOPs vs 100 (%)'] = (100 * (H3['GFLOPs'] / H3['GFLOPs'][0] - 1)).round(2)
print(H3.round(2)); H3.to_csv('../results/costo_vs_N.csv', index=False)